# OOP Opdracht: Library Management System
# DEEL 2 - Opdracht 5 t/m 8

**Deadline:** 28 November  

## Introductie

Welkom bij **Deel 2** van de Library Management System opdracht!

In Deel 1 heb je de basis classes gemaakt (Book, Member) en geleerd over:
- Instance methods
- Inheritance (EBook, AudioBook, PhysicalBook, StudentMember, PremiumMember)
- Getters en Setters (properties)
- Polymorfisme

In dit deel ga je je systeem verder uitbreiden met geavanceerde OOP concepten:
- **Abstract Base Classes & Interfaces** (Opdracht 5)
- **Composition** (Opdracht 6)
- **Iterators & Generators** (Opdracht 7)
- **Modules & Scope** (Opdracht 8)

**Belangrijk:** Je hebt je classes uit Deel 1 nodig! Kopieer deze naar dit notebook of importeer ze.

---

## Voorbereiding: Kopieer je Classes uit Deel 1

Voordat je begint met Deel 2, kopieer je **alle classes** die je in Deel 1 hebt gemaakt naar de cel hieronder:
- Book (met properties)
- EBook, AudioBook, PhysicalBook
- Member (met properties)
- StudentMember, PremiumMember

**Of** je kunt ze importeren als je ze in aparte .py files hebt gezet.

In [3]:
from datetime import datetime
from copy import deepcopy


In [4]:
# alle member-gerelateerde types

class Member:
    def __init__(self, name, memeber_id, email):
        self.name = name
        self.member_id = memeber_id
        self.__email = email
        self.__borrowed_books = []

    @property
    def max_books(self):
        return 5

    @property
    def email(self):
        return self.__email
    
    @email.setter
    def email(self, new_email):
        if "@" in new_email:
            self.__email = new_email
        else:
            print(f"Error: email must contain @")

    @property
    def borrowed_books(self):
        return deepcopy(self.__borrowed_books)

    def display_info(self):
        info = f"Member: {self.name}, id: {self.member_id}, email: {self.__email}"
        print(info)

    def get_borrowed_count(self):
        return len(self.borrowed_books)
    
    def can_borrow(self):
        if len(self.borrowed_books) < self.max_books:
            return True
        else:
            return False
    

class StudentMember(Member):

    def __init__(self, name, member_id, email, student_id, university):
        super().__init__(name, member_id, email)
        self.student_id = student_id
        self.university = university

    @property
    def max_books(self):
        return 3

    def display_info(self):
        super().display_info()
        print(f"{' '*8}student_id: {self.student_id}, university: {self.university}")


class PremiumMember(Member):
    def __init__(self, name, member_id, email, membership_expiry):
        super().__init__(name, member_id, email)
        self.membership_expiry = membership_expiry

    @property
    def max_books(self):
        return 10

    def is_expired(self):
        today = datetime.today().strftime('%Y-%m-%d')
        if self.membership_expiry <= today:
            return True
        else:
            return False

    def display_info(self):
        super().display_info()
        print(f"{' '*8}premium membership until: {self.membership_expiry}")

    def can_borrow(self):
        if self.get_borrowed_count() < self.max_books and not self.is_expired():
            return True
        else:
            return False

---

## Deel 5: Abstract Base Classes & Interfaces (60 minuten)

**Theorie:** ABC module, `@abstractmethod`, Interfaces, Abstract classes

### Opdracht 5.1: Abstract LibraryItem Base Class

Herstructureer je code met een abstract base class:

**Vereisten:**
- Importeer: `from abc import ABC, abstractmethod`
- Maak een abstract `LibraryItem` class (erft van `ABC`)
- Abstract methods die elke subclass moet implementeren:
  - `display_info()`: Toon item informatie
  - `get_type()`: Return het type item
  - `borrow()`: Leen logica
  - `return_item()`: Return logica
- Concrete methods:
  - `get_id()`: Return een unieke ID (implementeer in base class, gebruik isbn of ander attribuut)
- Laat `Book` (en zijn subclasses) van `LibraryItem` erven in plaats van direct

**Hint:** Je Book class wordt nu:
```python
class Book(LibraryItem):
    # ... rest van je code
```

**Test je code:**
```python
# item = LibraryItem()  # Zou fout moeten geven - kan niet instantiëren
book = PhysicalBook("Test", "Author", "ISBN123", 2020, "A-1", "New")
print(f"Type: {book.get_type()}")
print(f"ID: {book.get_id()}")
```

In [5]:
# OPDRACHT 5.1: Maak LibraryItem abstract base class
from abc import ABC, abstractmethod

class LibraryItem(ABC):

    @abstractmethod
    def display_info(self):
        """ Library item info """
        pass

    @abstractmethod
    def get_type(self):
        """ Type of a library item """
        pass

    @abstractmethod
    def borrow(self):
        """ Borrowing logic """
        pass

    @abstractmethod
    def return_item(self):
        """ Returning logic """
        pass

    def get_id(self):
        return getattr(self, "isbn", None)


class Book(LibraryItem):

    def __init__(self, title, author, isbn, publication_year):
        super().__init__()
        self.__isbn = isbn
        self.title = title
        self.author = author
        self.__publication_year = publication_year
        self.__available = True

    @property
    def isbn(self):
        return self.__isbn
    
    @property
    def publication_year(self):
        return self.__publication_year
    
    @property
    def available(self):
        return self.__available
    
    @available.setter
    def available(self, status):
        if isinstance(status, bool):
            self.__available = status
        else:
            print(f"Error: availability status must be True or False")
    
    @publication_year.setter
    def publication_year(self, year):
        today = datetime.today()
        if year >= 1000 and year <= today.year:
            self.__publication_year = year
        else:
            print(f"Error: year must be between 1000 and {today.year}")

    def display_info(self):
        info = f"Book: '{self.title}' by {self.author}, id: {super().get_id()}, published in {self.__publication_year}"
        print(info)

    def borrow(self):
        if self.__available == True:
            self.__available = False
        else:
            print("Warning: the book is unavailable")

    def return_item(self):
        self.__available = True


class EBook(Book):
    def __init__(self, title, author, isbn, publication_year, file_size, format):
        super().__init__(title, author, isbn, publication_year)
        self.file_size = file_size
        self.format = format
        self.download_counter = 0

    def display_info(self):
        super().display_info()
        print(f"{' '*6}file_size: {self.file_size}, format: {self.format}")

    def download(self):
        print(f"'{self.title}' had been downloaded")

    def borrow(self):
        super().borrow()
        self.download_counter += 1

    def get_type(self):
        return "EBook"
        
class AudioBook(Book):
    def __init__(self, title, author, isbn, publication_year, narrator, duration_minutes):
        super().__init__(title, author, isbn, publication_year)
        self.narrator = narrator
        self.duration_minutes = duration_minutes
        self.play_counter = 0

    def display_info(self):
        super().display_info()
        print(f"{' '*6}narrator: {self.narrator}, duration: {self.duration_minutes}")

    def play(self):
        print(f"'{self.title}' is playing now")

    def borrow(self):
        super().borrow()
        self.play_counter += 1
    
    def get_type(self):
        return "AudioBook"
    
    
class PhysicalBook(Book):
    def __init__(self, title, author, isbn, publication_year, shelf_location, condition):
        super().__init__(title, author, isbn, publication_year)
        self.shelf_location = shelf_location
        self.condition = condition

    def display_info(self):
        super().display_info()
        print(f"{' '*6}location: {self.shelf_location}, condition: {self.condition}")

    def get_type(self):
        return "PhysicalBook"

In [6]:
# Test code voor Opdracht 5.1
# Uncomment om te testen dat LibraryItem niet kan worden geïnstantieerd:
# item = LibraryItem()  # Zou TypeError moeten geven

book = PhysicalBook("Test Book", "Test Author", "ISBN123", 2020, "A-1", "New")
print(f"Type: {book.get_type()}")
print(f"ID: {book.get_id()}")
book.display_info()

Type: PhysicalBook
ID: ISBN123
Book: 'Test Book' by Test Author, id: ISBN123, published in 2020
      location: A-1, condition: New


### Opdracht 5.2: Searchable Interface

Maak een `Searchable` interface (abstract class) voor items die doorzoekbaar zijn:

**Vereisten:**
- Maak een abstract `Searchable` class (erft van `ABC`) met abstract methods:
  - `matches_search(search_term: str) -> bool`: Check of item matcht met zoekterm
  - `get_search_keywords() -> list`: Return lijst van zoekbare keywords
- Implementeer `Searchable` in je `Book` class (gebruik multiple inheritance):
  ```python
  class Book(LibraryItem, Searchable):
  ```
  - `matches_search()`: zoek in title, author, isbn (case-insensitive)
  - `get_search_keywords()`: return [title, author, isbn]
- Implementeer ook `Searchable` in je `Member` class:
  ```python
  class Member(Searchable):
  ```
  - `matches_search()`: zoek in name, member_id, email
  - `get_search_keywords()`: return [name, member_id, email]

**Test je code:**
```python
searchable_items = [
    PhysicalBook("Python Programming", "John Smith", "978-1234", 2023, "A-1", "New"),
    Member("John Doe", "M001", "john@email.com")
]

search_term = "John"
print(f"Searching for: {search_term}")
for item in searchable_items:
    if item.matches_search(search_term):
        print(f"Found: {item.get_search_keywords()}")
```

In [7]:
# OPDRACHT 5.2: Implementeer Searchable interface

class Searchable(ABC):
    """ Defines methods for searchable items """

@abstractmethod
def matches_search(search_term: str) -> bool:
    """ Checks if an item matchtes search_term """
    pass

@abstractmethod
def get_search_keywords() -> list:
    """ Returns list of searchable attributes """
    pass


class Book(LibraryItem, Searchable):

    def __init__(self, title, author, isbn, publication_year):
        super().__init__()
        self.__isbn = isbn
        self.title = title
        self.author = author
        self.__publication_year = publication_year
        self.__available = True

    @property
    def publication_year(self):
        return self.__publication_year
    
    @property
    def available(self):
        return self.__available
    
    @available.setter
    def available(self, status):
        if isinstance(status, bool):
            self.__available = status
        else:
            print(f"Error: availability status must be True or False")
    
    @publication_year.setter
    def publication_year(self, year):
        today = datetime.today()
        if year >= 1000 and year <= today.year:
            self.__publication_year = year
        else:
            print(f"Error: year must be between 1000 and {today.year}")

    def display_info(self):
        info = f"Book: '{self.title}' by {self.author}, id: {super().get_id()}, published in {self.__publication_year}"
        print(info)

    def borrow(self):
        if self.__available == True:
            self.__available = False
        else:
            print("Warning: the book is unavailable")

    def return_item(self):
        self.__available = True

    def get_search_keywords(self):
        return [self.title, self.author, self.get_id()]
    
    
    def matches_search(self, search_term: str) -> bool:
        term = search_term.casefold()
        for kw in self.get_search_keywords():
            if kw and term in str(kw).casefold():
                return True
        return False


class EBook(Book):
    def __init__(self, title, author, isbn, publication_year, file_size, format):
        super().__init__(title, author, isbn, publication_year)
        self.file_size = file_size
        self.format = format
        self.download_counter = 0

    def display_info(self):
        super().display_info()
        print(f"{' '*6}file_size: {self.file_size}, format: {self.format}")

    def download(self):
        print(f"'{self.title}' had been downloaded")

    def borrow(self):
        super().borrow()
        self.download_counter += 1

    def get_type(self):
        return "EBook"
        
class AudioBook(Book):
    def __init__(self, title, author, isbn, publication_year, narrator, duration_minutes):
        super().__init__(title, author, isbn, publication_year)
        self.narrator = narrator
        self.duration_minutes = duration_minutes
        self.play_counter = 0

    def display_info(self):
        super().display_info()
        print(f"{' '*6}narrator: {self.narrator}, duration: {self.duration_minutes}")

    def play(self):
        print(f"'{self.title}' is playing now")

    def borrow(self):
        super().borrow()
        self.play_counter += 1
    
    def get_type(self):
        return "AudioBook"
    
    
class PhysicalBook(Book):
    def __init__(self, title, author, isbn, publication_year, shelf_location, condition):
        super().__init__(title, author, isbn, publication_year)
        self.shelf_location = shelf_location
        self.condition = condition

    def display_info(self):
        super().display_info()
        print(f"{' '*6}location: {self.shelf_location}, condition: {self.condition}")

    def get_type(self):
        return "PhysicalBook"

In [8]:
class Member(Searchable):
    def __init__(self, name, memeber_id, email):
        self.name = name
        self.member_id = memeber_id
        self.__email = email
        self.__borrowed_books = []

    @property
    def max_books(self):
        return 5

    @property
    def email(self):
        return self.__email
    
    @email.setter
    def email(self, new_email):
        if "@" in new_email:
            self.__email = new_email
        else:
            print(f"Error: email must contain @")

    @property
    def borrowed_books(self):
        return deepcopy(self.__borrowed_books)

    def display_info(self):
        info = f"Member: {self.name}, id: {self.member_id}, email: {self.__email}"
        print(info)

    def get_borrowed_count(self):
        return len(self.borrowed_books)
    
    def can_borrow(self):
        if len(self.borrowed_books) < self.max_books:
            return True
        else:
            return False
        
    def get_search_keywords(self):
        return [self.name, self.member_id, self.email]

    def matches_search(self, search_term: str) -> bool:
        term = search_term.casefold()
        for kw in self.get_search_keywords():
            if kw and term in str(kw).casefold():
                return True
        return False

    
class StudentMember(Member):

    def __init__(self, name, member_id, email, student_id, university):
        super().__init__(name, member_id, email)
        self.student_id = student_id
        self.university = university

    @property
    def max_books(self):
        return 3

    def display_info(self):
        super().display_info()
        print(f"{' '*8}student_id: {self.student_id}, university: {self.university}")


class PremiumMember(Member):
    def __init__(self, name, member_id, email, membership_expiry):
        super().__init__(name, member_id, email)
        self.membership_expiry = membership_expiry

    @property
    def max_books(self):
        return 10

    def is_expired(self):
        today = datetime.today().strftime('%Y-%m-%d')
        if self.membership_expiry <= today:
            return True
        else:
            return False

    def display_info(self):
        super().display_info()
        print(f"{' '*8}premium membership until: {self.membership_expiry}")

    def can_borrow(self):
        if self.get_borrowed_count() < self.max_books and not self.is_expired():
            return True
        else:
            return False

In [9]:
# Test code voor Opdracht 5.2
searchable_items = [
    PhysicalBook("Python Programming", "John Smith", "978-1234", 2023, "A-1", "New"),
    Member("John Doe", "M001", "john@email.com")
]

search_term = "John"
print(f"Searching for: '{search_term}'")
for item in searchable_items:
    if item.matches_search(search_term):
        print(f"✓ Found: {item.get_search_keywords()}")

Searching for: 'John'
✓ Found: ['Python Programming', 'John Smith', None]
✓ Found: ['John Doe', 'M001', 'john@email.com']


---

## Deel 6: Composition (60 minuten)

**Theorie:** Composition over inheritance, "Has-a" relaties, Aggregation

### Opdracht 6.1: Transaction System

Maak een `Transaction` class voor het bijhouden van leen-transacties:

**Vereisten:**
- `Transaction` class met attributen:
  - `transaction_id` (str): Unieke transaction ID
  - `member` (Member object): Welk lid
  - `book` (Book object): Welk boek
  - `borrow_date` (str): Datum van lenen (formaat: "YYYY-MM-DD")
  - `return_date` (str of None): Datum van inleveren
  - `is_returned` (bool): Of het boek is ingeleverd (start False)
- Methods:
  - `__init__()`: Initialiseer transaction
  - `complete_return(return_date)`: Markeer als ingeleverd
  - `display_transaction()`: Toon transaction details
  - `calculate_days()`: Bereken aantal dagen geleend (simpele implementatie, mag geschat)

**Test je code:**
```python
book = PhysicalBook("Book Title", "Author", "ISBN", 2020, "A-1", "New")
member = Member("Alice", "M001", "alice@email.com")
transaction = Transaction("T001", member, book, "2025-01-01")
transaction.display_transaction()
transaction.complete_return("2025-01-15")
print(f"Days borrowed: {transaction.calculate_days()}")
```

In [10]:
# OPDRACHT 6.1: Implementeer Transaction class

class Transaction:
    def __init__(self, transaction_id, member, book, borrow_date="", return_date=""):
        self.transaction_id = transaction_id
        self.member = member
        self.book = book
        self.borrow_date = borrow_date
        self.return_date = return_date
        self.is_returned = False

    def complete_return(self, return_date):
        self.return_date = return_date
        self.is_returned = True

    def display_transaction(self):
        info = f"Book {self.book.title} by {self.book.author} is"
        if self.is_returned == True:
            info += f" returned on {self.return_date}"
        else:
            info += f" borrowed on {self.borrow_date}"
        return info

    def calculate_days(self):
        date_format = "%Y-%m-%d"
        borrowed = datetime.strptime(self.borrow_date, date_format)
        returned = datetime.strptime(self.return_date, date_format)
        return returned - borrowed



In [11]:
# Test code voor Opdracht 6.1
book = PhysicalBook("Book Title", "Author Name", "ISBN-001", 2020, "A-1", "New")
member = Member("Alice Johnson", "M001", "alice@email.com")
transaction = Transaction("T001", member, book, "2025-01-01")

print("New transaction:")
print(transaction.display_transaction())
transaction.complete_return("2025-01-15")
print("\nAfter return:")
print(transaction.display_transaction())
print(f"Total days borrowed: {transaction.calculate_days()}")

New transaction:
Book Book Title by Author Name is borrowed on 2025-01-01

After return:
Book Book Title by Author Name is returned on 2025-01-15
Total days borrowed: 14 days, 0:00:00


### Opdracht 6.2: Library Class (Composition)

Maak een centrale `Library` class die alles samenbrengt:

**Vereisten:**
- `Library` class bevat (composition):
  - `name` (str): Naam van de bibliotheek
  - `books` (list): Lijst van alle books in de bibliotheek (start leeg)
  - `members` (list): Lijst van alle members (start leeg)
  - `transactions` (list): Lijst van alle transactions (start leeg)
- Methods:
  - `__init__(name)`: Initialiseer library
  - `add_book(book)`: Voeg boek toe aan bibliotheek
  - `add_member(member)`: Voeg lid toe
  - `remove_book(isbn)`: Verwijder boek op basis van ISBN
  - `remove_member(member_id)`: Verwijder lid op basis van ID
  - `find_book(isbn)`: Vind en return boek op ISBN (of None)
  - `find_member(member_id)`: Vind en return lid op ID (of None)
  - `display_all_books()`: Toon alle boeken
  - `display_all_members()`: Toon alle leden

**Test je code:**
```python
library = Library("City Central Library")
library.add_book(PhysicalBook("1984", "Orwell", "ISBN1", 1949, "A-1", "Good"))
library.add_book(EBook("Python Guide", "Author", "ISBN2", 2023, 5.0, "PDF"))
library.add_member(Member("Alice", "M001", "alice@email.com"))

library.display_all_books()
library.display_all_members()

found = library.find_book("ISBN1")
print(f"\nFound book: {found.title if found else 'Not found'}")
```

In [12]:
# OPDRACHT 6.2: Implementeer Library class
class Library:
    def __init__ (self, name):
        self.name = name
        self.books = []
        self.members = []
        self.transactions = []

    
    def add_book(self, book):
        self.books.append(book)
        print(f"Book {book.title} has been added to the library.")

    def add_member(self, member):
        self.members.append(member)
        print(f"Member {member.name} has been added to the library.")

    def remove_book(self, isbn):
        for b in self.books:
            if b.get_id() == isbn:
                btitle = b.title
                self.books.remove(b)
                print(f"Book {btitle} has been removed from the library.")

    def remove_member(self, member_id):
        for m in self.members:
            if m.member_id == member_id:
                mname = m.name
                self.members.remove(m)
                print(f"Member {mname} has been removed from the library.")

    def find_book(self, isbn):
        """ Find and return book by ISBN (or None) """
        for b in self.books:
            if b.get_id() == isbn:
                return b

    def find_member(self, member_id):
        """ Find and return member by ID (or None) """
        for m in self.members:
            if m.member_id == member_id:
                return m

    def display_all_books(self):
        for b in self.books:
            b.display_info()

    def display_all_members(self):
        for m in self.members:
            m.display_info()


In [13]:
# Test code voor Opdracht 6.2
library = Library("City Central Library")

# Voeg boeken toe
library.add_book(PhysicalBook("1984", "George Orwell", "ISBN1", 1949, "A-1", "Good"))
library.add_book(EBook("Python Guide", "John Doe", "ISBN2", 2023, 5.0, "PDF"))
library.add_book(AudioBook("The Great Gatsby", "F. Scott Fitzgerald", "ISBN3", 1925, "Jake Gyllenhaal", 240))

# Voeg members toe
library.add_member(Member("Alice", "M001", "alice@email.com"))
library.add_member(StudentMember("Bob", "M002", "bob@uni.edu", "S001", "MIT"))

print("All Books:")
library.display_all_books()

print("\nAll Members:")
library.display_all_members()

print("\nSearching for book:")
found = library.find_book("ISBN1")
print(f"Found: {found.title if found else 'Not found'}")

Book 1984 has been added to the library.
Book Python Guide has been added to the library.
Book The Great Gatsby has been added to the library.
Member Alice has been added to the library.
Member Bob has been added to the library.
All Books:
Book: '1984' by George Orwell, id: None, published in 1949
      location: A-1, condition: Good
Book: 'Python Guide' by John Doe, id: None, published in 2023
      file_size: 5.0, format: PDF
Book: 'The Great Gatsby' by F. Scott Fitzgerald, id: None, published in 1925
      narrator: Jake Gyllenhaal, duration: 240

All Members:
Member: Alice, id: M001, email: alice@email.com
Member: Bob, id: M002, email: bob@uni.edu
        student_id: S001, university: MIT

Searching for book:
Found: Not found


### Opdracht 6.3: Borrowing System

Breid de `Library` class uit met leen-functionaliteit:

**Vereisten:**
- Voeg methods toe aan `Library`:
  - `borrow_book(member_id, isbn, borrow_date)`: 
    - Check of member bestaat (gebruik `find_member`)
    - Check of member `can_borrow()` (uit Deel 1)
    - Check of book bestaat (gebruik `find_book`) en available is
    - Creëer nieuwe Transaction
    - Update member's borrowed_books lijst
    - Update book's available status (via borrow() method)
    - Voeg transaction toe aan transactions lijst
    - Print success bericht
  - `return_book(member_id, isbn, return_date)`:
    - Vind de actieve transaction voor dit member en book
    - Complete de transaction (via `complete_return`)
    - Update member's borrowed_books lijst (verwijder boek)
    - Update book's available status (via `return_book()` method)
    - Print success bericht
  - `get_member_transactions(member_id)`: Return lijst van alle transactions van een lid
  - `get_active_transactions()`: Return lijst van nog niet ingeleverde transactions

**Test je code:**
```python
library = Library("Test Library")
book = PhysicalBook("Test Book", "Author", "ISBN1", 2020, "A-1", "New")
member = Member("Alice", "M001", "alice@email.com")
library.add_book(book)
library.add_member(member)

library.borrow_book("M001", "ISBN1", "2025-01-01")
print(f"Book available: {book.available}")

library.return_book("M001", "ISBN1", "2025-01-15")
print(f"Book available: {book.available}")

transactions = library.get_member_transactions("M001")
print(f"Total transactions for member: {len(transactions)}")
```

In [14]:
# OPDRACHT 6.3: Breid Library uit met borrowing system
# Kopieer je Library class en voeg de nieuwe methods toe

class Library:
    def __init__ (self, name):
        self.name = name
        self.books = []
        self.members = []
        self.transactions = []

    
    def add_book(self, book):
        self.books.append(book)
        print(f"Book {book.title} has been added to the library.")

    def add_member(self, member):
        self.members.append(member)
        print(f"Member {member.name} has been added to the library.")

    def remove_book(self, isbn):
        for b in self.books:
            if b.get_id() == isbn:
                btitle = b.title
                self.books.remove(b)
                print(f"Book {btitle} has been removed from the library.")

    def remove_member(self, member_id):
        for m in self.members:
            if m.member_id == member_id:
                mname = m.name
                self.members.remove(m)
                print(f"Member {mname} has been removed from the library.")

    def find_book(self, isbn):
        """ Find and return book by ISBN (or None) """
        for b in self.books:
            if b.get_id() == isbn:
                return b

    def find_member(self, member_id):
        """ Find and return member by ID (or None) """
        for m in self.members:
            if m.member_id == member_id:
                return m

    def display_all_books(self):
        for b in self.books:
            b.display_info()

    def display_all_members(self):
        for m in self.members:
            m.display_info()

    def borrow_book(self, member_id, isbn, borrow_date):
        """ Borrowing actions """
        member = self.find_member(member_id)
        book = self.find_book(isbn)
        if not member or not member.can_borrow():
            print("No such member or the member cannot borrow!")
            return
        if not book or not book.available:
            print("No such book or the book is anavailable!")
            return
        transaction = Transaction("T001", member, book, borrow_date)
        self.transactions.append(transaction)
        member._Member__borrowed_books.append(book)
        book.borrow()
        print(f"The book {book.title} has been successfully borrowed by {member.name}")
        
    def return_book(self, member_id, isbn, return_date):
        """ Returning actions """
        for t in self.get_member_transactions(member_id):
            book = t.book
            if book.get_id() == isbn and not t.return_date:
                t.complete_return(return_date)
        member = self.find_member(member_id)
        member._Member__borrowed_books.remove(book)
        book.return_item()
        print(f"The book {book.title} has been successfully returned by {member.name}")
            
    def get_member_transactions(self, member_id):
        """ Returns the list of all transactions of a member """
        m_transactions = []
        for t in self.transactions:
            mem = t.member
            if mem.member_id == member_id:
                m_transactions.append(t)
        return m_transactions

    def get_active_transactions(self):
        """ Returns the list of all borrowed and not yet returned transactions """
        a_transactions = []
        for t in self.transactions:
            if not t.return_date:
                a_transactions.append(t)
        return a_transactions


In [15]:
# Test code voor Opdracht 6.3
library = Library("Test Library")
book = PhysicalBook("Test Book", "Author", "ISBN1", 2020, "A-1", "New")
member = Member("Alice", "M001", "alice@email.com")
library.add_book(book)
library.add_member(member)

print("=" * 50)
print("Testing Borrow System")
print("=" * 50)

print(f"\nInitial - Book available: {book.available} and Member can borrow: {member.can_borrow()}")

library.borrow_book("M001", "ISBN1", "2025-01-01")
print(f"After borrow - Book available: {book.available}")

library.return_book("M001", "ISBN1", "2025-01-15")
print(f"After return - Book available: {book.available}")

transactions = library.get_member_transactions("M001")
print(f"\nTotal transactions for member: {len(transactions)}")

active = library.get_active_transactions()
print(f"Active transactions: {len(active)}")

Book Test Book has been added to the library.
Member Alice has been added to the library.
Testing Borrow System

Initial - Book available: True and Member can borrow: True
No such book or the book is anavailable!
After borrow - Book available: True


UnboundLocalError: cannot access local variable 'book' where it is not associated with a value

---

## Deel 7: Iterators & Generators (45 minuten)

**Theorie:** `__iter__()`, `__next__()`, `yield`, Generator expressions

### Opdracht 7.1: Library Iterator

Maak je `Library` class itereerbaar:

**Vereisten:**
- Implementeer `__iter__()` in de `Library` class
  - Return `self`
  - Initialiseer een counter (bijv. `self._iter_index = 0`)
- Implementeer `__next__()` in de `Library` class
  - Return het volgende book uit de books lijst
  - Verhoog de counter
  - Raise `StopIteration` wanneer alle books zijn doorlopen
- Iteratie moet over alle books in de bibliotheek gaan

**Test je code:**
```python
library = Library("Test Library")
library.add_book(PhysicalBook("Book1", "Author1", "ISBN1", 2020, "A-1", "New"))
library.add_book(EBook("Book2", "Author2", "ISBN2", 2021, 5.0, "PDF"))
library.add_book(AudioBook("Book3", "Author3", "ISBN3", 2022, "Narrator", 180))

print("Iterating through library:")
for book in library:
    print(f"- {book.title} ({book.get_type()})")
```

In [ ]:
# OPDRACHT 7.1: Maak Library itereerbaar
# Voeg __iter__ en __next__ methods toe aan je Library class

class Library:
    def __init__ (self, name):
        self.name = name
        self.books = []
        self.members = []
        self.transactions = []
        self._iter_index = -1

    
    def add_book(self, book):
        self.books.append(book)
        print(f"Book {book.title} has been added to the library.")

    def add_member(self, member):
        self.members.append(member)
        print(f"Member {member.name} has been added to the library.")

    def remove_book(self, isbn):
        for b in self.books:
            if b.get_id() == isbn:
                btitle = b.title
                self.books.remove(b)
                print(f"Book {btitle} has been removed from the library.")

    def remove_member(self, member_id):
        for m in self.members:
            if m.member_id == member_id:
                mname = m.name
                self.members.remove(m)
                print(f"Member {mname} has been removed from the library.")

    def find_book(self, isbn):
        """ Find and return book by ISBN (or None) """
        for b in self.books:
            if b.get_id() == isbn:
                return b

    def find_member(self, member_id):
        """ Find and return member by ID (or None) """
        for m in self.members:
            if m.member_id == member_id:
                return m

    def display_all_books(self):
        for b in self.books:
            b.display_info()

    def display_all_members(self):
        for m in self.members:
            m.display_info()

    def borrow_book(self, member_id, isbn, borrow_date):
        """ Borrowing actions """
        member = self.find_member(member_id)
        book = self.find_book(isbn)
        if not member or not member.can_borrow():
            print("No such member or the member cannot borrow!")
            return
        if not book or not book.available:
            print("No such book or the book is anavailable!")
            return
        transaction = Transaction("T001", member, book, borrow_date)
        self.transactions.append(transaction)
        member._Member__borrowed_books.append(book)
        book.borrow()
        print(f"The book {book.title} has been successfully borrowed by {member.name}")
        
    def return_book(self, member_id, isbn, return_date):
        """ Returning actions """
        for t in self.get_member_transactions(member_id):
            book = t.book
            if book.get_id() == isbn and not t.return_date:
                t.complete_return(return_date)
        member = self.find_member(member_id)
        member._Member__borrowed_books.remove(book)
        book.return_item()
        print(f"The book {book.title} has been successfully returned by {member.name}")
            
    def get_member_transactions(self, member_id):
        """ Returns the list of all transactions of a member """
        m_transactions = []
        for t in self.transactions:
            mem = t.member
            if mem.member_id == member_id:
                m_transactions.append(t)
        return m_transactions

    def get_active_transactions(self):
        """ Returns the list of all borrowed and not yet returned transactions """
        a_transactions = []
        for t in self.transactions:
            if not t.return_date:
                a_transactions.append(t)
        return a_transactions
    
    def __iter__(self):
        return self
    
    def __next__(self):
        self._iter_index += 1
        if self._iter_index == len(self.books):
            raise StopIteration
        return self.books[self._iter_index]


In [ ]:
# Test code voor Opdracht 7.1
library = Library("Test Library")
library.add_book(PhysicalBook("Book1", "Author1", "ISBN1", 2020, "A-1", "New"))
library.add_book(EBook("Book2", "Author2", "ISBN2", 2021, 5.0, "PDF"))
library.add_book(AudioBook("Book3", "Author3", "ISBN3", 2022, "Narrator", 180))

print("Iterating through library:")
for book in library:
    print(f"- {book.title} ({book.get_type()})")

Book Book1 has been added to the library.
Book Book2 has been added to the library.
Book Book3 has been added to the library.
Iterating through library:


TypeError: 'Library' object is not iterable

### Opdracht 7.2: Generator Methods

Voeg generator methods toe aan de `Library` class:

**Vereisten:**
- `available_books()`: Generator die alleen beschikbare boeken yield
  ```python
  def available_books(self):
      for book in self.books:
          if book.available:
              yield book
  ```
- `books_by_author(author_name)`: Generator die boeken van een specifieke auteur yield (case-insensitive)
- `books_by_type(book_type)`: Generator die boeken van een specifiek type yield ("EBook", "AudioBook", "PhysicalBook")
- `active_members()`: Generator die members met geleende boeken yield

**Test je code:**
```python
library = Library("Test Library")
# ... voeg boeken toe ...

print("Available books:")
for book in library.available_books():
    print(f"- {book.title}")

print("\nBooks by George Orwell:")
for book in library.books_by_author("George Orwell"):
    print(f"- {book.title}")

print("\nE-Books:")
for book in library.books_by_type("EBook"):
    print(f"- {book.title}")
```

In [ ]:
# OPDRACHT 7.2: Implementeer generator methods

class Library:
    def __init__ (self, name):
        self.name = name
        self.books = []
        self.members = []
        self.transactions = []

    def add_book(self, book):
        self.books.append(book)
        print(f"Book {book.title} has been added to the library.")

    def add_member(self, member):
        self.members.append(member)
        print(f"Member {member.name} has been added to the library.")

    def remove_book(self, isbn):
        for b in self.books:
            if b.get_id() == isbn:
                btitle = b.title
                self.books.remove(b)
                print(f"Book {btitle} has been removed from the library.")

    def remove_member(self, member_id):
        for m in self.members:
            if m.member_id == member_id:
                mname = m.name
                self.members.remove(m)
                print(f"Member {mname} has been removed from the library.")

    def find_book(self, isbn):
        """ Find and return book by ISBN (or None) """
        for b in self.books:
            if b.get_id() == isbn:
                return b

    def find_member(self, member_id):
        """ Find and return member by ID (or None) """
        for m in self.members:
            if m.member_id == member_id:
                return m

    def display_all_books(self):
        for b in self.books:
            b.display_info()

    def display_all_members(self):
        for m in self.members:
            m.display_info()

    def borrow_book(self, member_id, isbn, borrow_date):
        """ Borrowing actions """
        member = self.find_member(member_id)
        book = self.find_book(isbn)
        if not member or not member.can_borrow():
            print("No such member or the member cannot borrow!")
            return
        if not book or not book.available:
            print("No such book or the book is anavailable!")
            return
        transaction = Transaction("T001", member, book, borrow_date)
        self.transactions.append(transaction)
        member._Member__borrowed_books.append(book)
        book.borrow()
        print(f"The book {book.title} has been successfully borrowed by {member.name}")
        
    def return_book(self, member_id, isbn, return_date):
        """ Returning actions """
        for t in self.get_member_transactions(member_id):
            book = t.book
            if book.get_id() == isbn and not t.return_date:
                t.complete_return(return_date)
        member = self.find_member(member_id)
        member._Member__borrowed_books.remove(book)
        book.return_item()
        print(f"The book {book.title} has been successfully returned by {member.name}")
            
    def get_member_transactions(self, member_id):
        """ Returns the list of all transactions of a member """
        m_transactions = []
        for t in self.transactions:
            mem = t.member
            if mem.member_id == member_id:
                m_transactions.append(t)
        return m_transactions

    def get_active_transactions(self):
        """ Returns the list of all borrowed and not yet returned transactions """
        a_transactions = []
        for t in self.transactions:
            if not t.return_date:
                a_transactions.append(t)
        return a_transactions
    
    def __iter__(self):
        self.__index = -1
        return self
    
    def __next__(self):
        self.__index += 1
        if self.__index >= len(self.books):
            raise StopIteration
        return self.books[self.__index]
    
    def available_books(self):
      for book in self.books:
          if book.available:
              yield book

    def books_by_author(self, author_name):
        for book in self.books:
            if book.author == author_name:
                yield book

    def books_by_type(self, book_type):
        for book in self.books:
            if book.__class__.__name__ == book_type:
                yield book

    def active_members(self):
        """ Members with borrowed books """
        for m in self.members:
            if m.borrowed_books:
                yield m

In [ ]:
# Test code voor Opdracht 7.2
library = Library("Test Library")
library.add_book(PhysicalBook("1984", "George Orwell", "ISBN1", 1949, "A-1", "Good"))
library.add_book(EBook("Python Guide", "John Doe", "ISBN2", 2023, 5.0, "PDF"))
library.add_book(EBook("Animal Farm", "George Orwell", "ISBN3", 1945, 2.5, "EPUB"))
library.add_book(AudioBook("Great Gatsby", "F. Scott Fitzgerald", "ISBN4", 1925, "Narrator", 240))

library.add_member(Member("Alice", "M001", "alice@email.com"))

# Leen een boek om available te testen
library.books[0].borrow()

print("Available books:")
for book in library.available_books():
    print(f"- {book.title}")

print("\nBooks by George Orwell:")
for book in library.books_by_author("George Orwell"):
    print(f"- {book.title}")

print("\nE-Books:")
for book in library.books_by_type("EBook"):
    print(f"- {book.title}")

# Test active members:
library.borrow_book("M001", "ISBN3", "2025-01-01")
print("\nActive members:")
for m in library.active_members():
    print(f"- {m.name}")

Book 1984 has been added to the library.
Book Python Guide has been added to the library.
Book Animal Farm has been added to the library.
Book Great Gatsby has been added to the library.
Member Alice has been added to the library.
Available books:
- Python Guide
- Animal Farm
- Great Gatsby

Books by George Orwell:
- 1984
- Animal Farm

E-Books:
- Python Guide
- Animal Farm
The book Animal Farm has been successfully borrowed by Alice

Active members:
- Alice


### Opdracht 7.3: Custom Book Collection Iterator

Maak een aparte `BookCollection` class met iterator functionaliteit:

**Vereisten:**
- `BookCollection` class die een lijst van books beheert
- Implementeer `__iter__()` en `__next__()`
- Extra feature: optie om te itereren in omgekeerde volgorde
- Method `__len__()`: return aantal boeken in de collectie
- Method `__getitem__(index)`: toegang tot boeken via index (zoals een lijst)
- Method `add_book(book)`: voeg boek toe aan collectie
- Method `sort_by_title()`: sorteer boeken alfabetisch op titel

**Test je code:**
```python
collection = BookCollection()
collection.add_book(PhysicalBook("Zebra Book", "Author", "ISBN1", 2020, "A-1", "New"))
collection.add_book(PhysicalBook("Apple Book", "Author", "ISBN2", 2021, "A-2", "New"))
collection.add_book(PhysicalBook("Mango Book", "Author", "ISBN3", 2022, "A-3", "New"))

print(f"Collection size: {len(collection)}")
print(f"First book: {collection[0].title}")

print("\nOriginal order:")
for book in collection:
    print(f"- {book.title}")

collection.sort_by_title()
print("\nSorted by title:")
for book in collection:
    print(f"- {book.title}")
```

In [ ]:
# OPDRACHT 7.3: Implementeer BookCollection iterator class

class BookCollection:

    def __init__(self, reverse = False):
        self.books = []
        self.reverse = reverse
    
    def __iter__(self):
        self.__index = -1
        return self

    def __next__(self):
        self.__index += 1
        if self.__index >= len(self.books):
            raise StopIteration
        if self.reverse:
            # yield from the end when reverse=True
            return self.books[len(self.books) - 1 - self.__index]
        return self.books[self.__index]

    def __len__(self):
        return len(self.books)

    def __getitem__(self, index):
        return self.books[index]

    def add_book(self, book):
        self.books.append(book)
        
    def sort_by_title(self, reverse=False):
        """Sort the collection in-place by book.title (ascending by default)."""
        self.books.sort(key=lambda x: x.title, reverse=reverse)
    

In [ ]:
# Test code voor Opdracht 7.3
collection = BookCollection()
collection.add_book(PhysicalBook("Zebra Book", "Author Z", "ISBN1", 2020, "A-1", "New"))
collection.add_book(PhysicalBook("Apple Book", "Author A", "ISBN2", 2021, "A-2", "New"))
collection.add_book(PhysicalBook("Mango Book", "Author M", "ISBN3", 2022, "A-3", "New"))

print(f"Collection size: {len(collection)}")
print(f"First book: {collection[0].title}")
print(f"Last book: {collection[-1].title}")

print("\nOriginal order:")
for book in collection:
    print(f"- {book.title}")

collection.sort_by_title(reverse=True)
print("\nSorted by title:")
for book in collection:
    print(f"- {book.title}")

Collection size: 3
First book: Zebra Book
Last book: Mango Book

Original order:
- Zebra Book
- Apple Book
- Mango Book

Sorted by title:
- Zebra Book
- Mango Book
- Apple Book


---

## Deel 8: Scope & Modules (60 minuten)

**Theorie:** Modules, Imports, Namespaces, `__name__`, Package structure

### Opdracht 8.1: Module Structuur

Organiseer je code in aparte Python modules (`.py` files):

**Vereisten:**
Maak de volgende files in je workspace:

1. **`library_items.py`**:
   - LibraryItem (abstract class)
   - Book (base class)
   - EBook, AudioBook, PhysicalBook

2. **`members.py`**:
   - Member (base class)
   - StudentMember, PremiumMember

3. **`transactions.py`**:
   - Transaction class

4. **`library.py`**:
   - Library class
   - Importeer: `from library_items import Book, EBook, AudioBook, PhysicalBook`
   - Importeer: `from members import Member`
   - Importeer: `from transactions import Transaction`

5. **`interfaces.py`**:
   - Searchable interface (abstract class)

6. **`book_collection.py`**:
   - BookCollection class

**Tip:** In elk .py file, voeg toe aan het einde:
```python
if __name__ == "__main__":
    # Test code voor dit specifieke module
    print("Testing [module name]...")
```

**Test je imports hieronder:**

In [ ]:
# OPDRACHT 8.1: Deze opdracht doe je in aparte .py files
# Maak de files aan in je workspace volgens de structuur hierboven

# Test hier of je imports werken:
from library_items import Book, EBook, AudioBook, PhysicalBook
from members import Member, StudentMember, PremiumMember
from transactions import Transaction
from library import Library
from interfaces import Searchable
from book_collection import BookCollection

# Test een class:
book = PhysicalBook("Test", "Author", "ISBN", 2020, "A-1", "New")
print(f"Successfully imported: {book.title}")

Successfully imported: Test


### Opdracht 8.2: Configuration Module

Maak een configuration module voor library settings:

**Vereisten:**
- Maak `config.py` met constanten:
  ```python
  # Library Configuration Settings
  
  MAX_BORROW_DAYS = 14  # Maximum dagen om te lenen
  DEFAULT_MAX_BOOKS = 5  # Standaard max aantal boeken
  STUDENT_MAX_BOOKS = 3  # Max voor studenten
  PREMIUM_MAX_BOOKS = 10  # Max voor premium members
  LATE_FEE_PER_DAY = 0.50  # Boete per dag te laat (euro)
  LIBRARY_NAME = "Central Library"  # Standaard naam
  LIBRARY_EMAIL = "info@library.com"
  LIBRARY_PHONE = "123-456-7890"
  ```
- Gebruik deze configuratie in je Library en Member classes
- Pas `StudentMember.max_books` aan om `config.STUDENT_MAX_BOOKS` te gebruiken
- Pas `PremiumMember.max_books` aan om `config.PREMIUM_MAX_BOOKS` te gebruiken
- Voeg een method `calculate_late_fee()` toe aan Transaction class die config gebruikt

**Test je code:**
```python
import config
print(f"Max borrow days: {config.MAX_BORROW_DAYS}")
print(f"Late fee per day: €{config.LATE_FEE_PER_DAY}")
```

In [ ]:
# OPDRACHT 8.2: Maak config.py
# Test hier je config imports:

import config
print(f"Library name: {config.LIBRARY_NAME}")
print(f"Max borrow days: {config.MAX_BORROW_DAYS}")
print(f"Student max books: {config.STUDENT_MAX_BOOKS}")
print(f"Premium max books: {config.PREMIUM_MAX_BOOKS}")
print(f"Late fee per day: €{config.LATE_FEE_PER_DAY}")

Library name: Central Library
Max borrow days: 14
Student max books: 3
Premium max books: 10
Late fee per day: €0.5


### Opdracht 8.3: Utility Module

Maak een utilities module met helper functies:

**Vereisten:**
- Maak `utils.py` met functies:
  ```python
  def generate_id(prefix, counter=[0]):
      """Genereer unieke ID met prefix (bijv. 'M001', 'B001')"""
      counter[0] += 1
      return f"{prefix}{counter[0]:03d}"
  
  def validate_email(email):
      """Valideer email formaat (verbeterde versie)"""
      # Implementeer betere email validatie
      pass
  
  def format_date(date_string):
      """Format datum naar consistent formaat"""
      pass
  
  def calculate_days_between(date1, date2):
      """Bereken dagen tussen twee datums (strings in YYYY-MM-DD)"""
      pass
  
  def validate_isbn(isbn):
      """Check of ISBN geldig formaat heeft"""
      pass
  ```
- Implementeer alle functies
- Gebruik deze utilities in je classes waar relevant
- Voeg een `if __name__ == "__main__":` block toe om utilities te testen

**Test je code:**
```python
from utils import generate_id, validate_email, validate_isbn

print(generate_id("M"))  # M001
print(generate_id("M"))  # M002
print(generate_id("B"))  # B001
print(validate_email("test@email.com"))  # True
print(validate_isbn("978-1234567890"))  # True
```

In [ ]:
# OPDRACHT 8.3: Maak utils.py en test het hier
from utils import generate_id, validate_email, validate_isbn, calculate_days_between

print("Testing utility functions:")
print(f"Generated ID: {generate_id('M')}")
print(f"Generated ID: {generate_id('M')}")
print(f"Generated ID: {generate_id('B')}")

print(f"Valid email: {validate_email('test@email.com')}")
print(f"Invalid email: {validate_email('invalid-email')}")
print(f"Valid ISBN: {validate_isbn('978-1234567890')}")
print(f"Days between: {calculate_days_between('2025-01-01', '2025-01-15')}")
print(f"Days between: {calculate_days_between('2025.01.10', '2025.01.09')}")

Testing utility functions:
Generated ID: M001
Generated ID: M002
Generated ID: B003
Valid email: True
Invalid email: False
Valid ISBN: True
Days between: 14
Days between: 1


---

## Einde van Deel 2

**Gefeliciteerd!** Je hebt Deel 2 voltooid!

### Wat heb je in Deel 2 geleerd?

✅ **Abstract Base Classes & Interfaces**: Contract enforcement met abstract methods  
✅ **Composition**: Library bevat Books, Members en Transactions  
✅ **Iterators**: Library is itereerbaar, custom BookCollection iterator  
✅ **Generators**: Efficiënte filtering met yield  
✅ **Modules**: Code organisatie in herbruikbare componenten  
✅ **Configuration**: Centrale configuratie management  
✅ **Utilities**: Herbruikbare helper functies  

### Je hebt nu:

- Een volledige Library class met borrow/return systeem
- Transaction tracking
- Searchable interface voor Books en Members
- Itereerbare collections
- Generator methods voor filtering
- Georganiseerde module structuur
- Configuration en utility modules

### Volgende stap: Deel 3 

---